# 第91章 K近邻模型（KNN）

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 6 / 34 步：扩展监督/无监督模型工具箱**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 逻辑回归分类  →  **本章任务：** K近邻模型（KNN）  →  **下一步：** 决策树
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

数据里常常藏着“物以类聚”的规律——特征相似的对象，往往属于同一类别，或被相似的数值预测。K 近邻（KNN）正是利用这一点：要判断一个新样本，就去看它周围离得最近的几个已知样本怎么说，让“身边人”替它投票。这种办法不需要假设复杂的数学形式，却能快速给出一个靠谱的基准，帮我们判断更复杂的模型是不是真的更好。这一章就用 Wine 数据看看 KNN 怎么工作，以及为什么必须给特征做缩放、邻居数 k 该怎么选。


## 本章目标

学完本章，你将能够：

- **理解**：理解「K近邻模型（KNN）」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「K近邻模型（KNN）」的关键输出指标。
- **迁移**：能把「K近邻模型（KNN）」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念
**背景引入**：数据里常常藏着“物以类聚”的规律——特征相似的对象，往往属于同一类别，或被相似的数值预测。K 近邻（KNN）正是利用这一点：要判断一个新样本，就去看它周围离得最近的几个已知样本怎么说，让“身边人”替它投票。这种办法不需要假设复杂的数学形式，却能快速给出一个靠谱的基准，帮我们判断更复杂的模型是不是真的更好。这一章就用 Wine 数据看看 KNN 怎么工作，以及为什么必须给特征做缩放、邻居数 k 该怎么选。

- k 小时边界灵活、方差较高
- k 大时边界平滑、偏差较高
- 预测成本随训练样本增长
- 距离模型对无关特征和量纲敏感（打个比方：身高用“米”、体重用“公斤”，数字大小天差地别；算“谁离我近”时会被数值大的那个特征牵着走，要先用同一把“尺子”统一。）


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| Wine 数据与缩放 | `raw.score()`、`scaled.score()`、`.fit()` | Wine 的 13 个化学特征量纲差异明显，适合展示缩放的重要性。 | 未缩放直接计算距离 |
| 比较邻居数 | `rows.append()`、`m.score()`、`pd.DataFrame()`、`.fit()` | 在固定切分上观察 k 的影响；正式选择应使用交叉验证。 | 使用训练准确率选择 k |


## 例 1｜Wine 数据与缩放

Wine 的 13 个化学特征量纲差异明显，适合展示缩放的重要性。


<!-- math-foundation:chapter-91 -->
### 数学推导｜KNN 由距离和邻居投票定义

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜计算查询点与每个训练样本的距离。** 例如欧氏距离 $d_i=\lVert x-x_i\rVert_2$。

**第 2 步｜选出距离最小的 $k$ 个索引。** 

$$
N_k(x)=\operatorname*{arg\,min}_{S:|S|=k}\sum_{i\in S}d_i
$$

**第 3 步｜汇总邻居标签。** 分类概率可估计为 $\hat p(y=c\mid x)=\sum_{i\in N_k(x)}\mathbf1(y_i=c)/k$，再选择概率最大的类别。$k$ 小时边界灵活但波动大，$k$ 大时更平滑但可能欠拟合。

**把上面的关系收束为本章计算式：**

$$
d(x,z)=\sqrt{\sum_j(x_j-z_j)^2},\qquad \hat{y}=\operatorname{mode}\{y_i:i\in N_k(x)\}
$$

**符号解释：** $N_k(x)$ 是离样本 $x$ 最近的 $k$ 个训练样本。

**代码对应：** 标准化后比较不同 `n_neighbors`，并通过验证集选择 $k$。

**使用边界：** 高维空间距离会退化；类别不平衡时多数类可能主导投票。


In [ ]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

data = load_wine(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=80
)
raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
scaled = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=5)
).fit(X_train, y_train)
print(
    "未缩放/缩放:",
    round(raw.score(X_test, y_test), 3),
    round(scaled.score(X_test, y_test), 3),
)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：示例里把 `n_neighbors=5` 用在 Wine 数据的测试集上。现在把邻居数改成 7，重新训练并输出“未缩放 / 缩放”两个准确率，观察只改这一个参数带来的变化。距离投票的邻居变多后，边界通常更平滑、方差更低，但也要留意是否会欠拟合。启动单元格后再对照下一格，把你能解释的变化写进笔记。


In [ ]:
try:
    # 请在下方补全代码：把示例 1 中的 n_neighbors 从 5 改成 7，
    # 训练并输出未缩放与缩放后的测试准确率。
    # k = ____  # 1) 把这里的邻居数改成 7   # ← 补全后取消注释
    raw_k = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    scaled_k = make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=k)
    ).fit(X_train, y_train)
    print(
        "k=7 未缩放/缩放：",
        round(raw_k.score(X_test, y_test), 3),
        round(scaled_k.score(X_test, y_test), 3),
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜比较邻居数

在固定切分上观察 k 的影响；正式选择应使用交叉验证。


In [ ]:
rows = []
for k in [1, 3, 5, 9, 15, 25]:
    m = make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=k)
    ).fit(X_train, y_train)
    rows.append([k, m.score(X_train, y_train), m.score(X_test, y_test)])
scores = pd.DataFrame(rows, columns=["k", "train", "test"])
display(scores)


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 未缩放直接计算距离
- 使用训练准确率选择 k
- 在高维稀疏数据中忽视维度灾难
- 大数据上忽略预测延迟


## 练习与作业

1. 用 weights='distance' 比较 k=3、5、9
2. 选择测试得分最高配置
3. 记录与均匀投票的差异

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 91.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“用 weights='distance' 比较 k=3、5、9”。
2. **独立完成**：不复制示例代码，完成“选择测试得分最高配置”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“记录与均匀投票的差异”，用一两句话说明你修改了什么。

### 91.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 91.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

掌握 KNN 的距离投票原理、特征缩放要求，以及邻居数 k 对欠拟合和过拟合的影响。


### 你已经掌握

- 训练 KNeighborsClassifier
- 解释距离和邻居投票
- 使用标准化避免量纲主导
- 通过验证比较不同 k


### 需要注意

- 未缩放直接计算距离
- 使用训练准确率选择 k
- 在高维稀疏数据中忽视维度灾难
- 大数据上忽略预测延迟


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
k = 7
raw_k = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
scaled_k = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=k)
).fit(X_train, y_train)
raw_score_k = raw_k.score(X_test, y_test)
scaled_score_k = scaled_k.score(X_test, y_test)
print("k=7 未缩放/缩放：", round(raw_score_k, 3), round(scaled_score_k, 3))


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
practice_scores = {}
for k in [3, 5, 9]:
    m = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k, weights="distance"),
    ).fit(X_train, y_train)
    practice_scores[k] = m.score(X_test, y_test)
print(practice_scores)
